<img src="https://ensai.fr/wp-content/uploads/2019/05/Ensai-logo.png" style="padding-right:10px;width:140px;float:left"></td> 
<h2 style="white-space: nowrap">Optimisation et méthodes numériques</h2>
<h3 style="white-space: nowrap">Lab 2: Support Vector Machines (SVM)</h3>

<hr style="clear:both">
<p style="font-size:0.85em; margin:0px"><b>Authors</b>: 
    <a href="mailto:sebastien.herbreteau@ensai.fr">Sébastien Herbreteau</a> and
    <a href="mailto:clement.elvira@centralesupelec.fr">Clément Elvira</a>.
</p>
<p style="font-size:0.85em; margin:2px; text-align:justify"> 

<u>**Short description:**</u> In this lab, you will solve the SVM optimization problem using **gradient descent**-based algorithms. As strong duality holds, you will explore both the **primal** and the **dual** approach. Finally, you will apply it to **binary classification** in *machine learning*.

## Imports

In the next cell we import Python libraries we will use throughout the lab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from utilities import project_onto_C

## <a id="ToC"></a>Table of contents
1. [Mathematical background](#1.-Mathematical-background)
    1. [The primal problem](#1.A.-The-primal-problem)
    2. [The dual problem](#1.B.-The-dual-problem)
2. [Breast Cancer Dataset](#2.-Breast-Cancer-Dataset)
3. [Solving the primal problem by gradient descent](#3.-Solving-the-primal-problem-by-gradient-descent)
4. [Solving the dual problem by projected gradient descent](#4.-Solving-the-dual-problem-by-projected-gradient-descent)
5. [Bonus: Sequential minimal optimization](#5.-Bonus:-Sequential-minimal-optimization)

## 1. Mathematical background
[Back to table of contents](#ToC)


Consider a task where one has to design a binary classifier, $\textit{i.e.}$ a function
\begin{equation*}
	h: \mathbb{R}^D \mapsto \{-1,+1\}
\end{equation*}
that takes a vector as input (referred to as features) and outputs a label in $\{-1,+1\}$ (think of "cat" and "dog").
Before the era of deep learning, predictors based on a linear separating hyperplane were popular. 
The idea was to parameterize the binary classifier with a vector $\mathbf{w}\in\mathbb{R}^{D}$ and a scalar $b\in\mathbb{R}$ such that the resulting (parameterized) function is
\begin{equation*} %\label{eq:linear predictor}
	h_{\mathbf{w}, b}: \mathbf{x} \in \mathbb{R}^{D} \mapsto \operatorname{sign}(\mathbf{w}^\top \mathbf{x} + b).
\end{equation*}
The vector $\mathbf{w}$ is called the $\textit{weight vector}$ and the scalar $b$ is the $\textit{bias}$.


But how to choose the parameters $(\mathbf{w},b) \in\mathbb{R}^{D} \times \mathbb{R}$ to build a good binary classifier $h_{\mathbf{w}, b}$? Assuming that one has a training set $\{(\mathbf{x}_i,y_i)\}_{i=1}^{n}$, where $y_i\in\{-1,+1\}$ is the label of the $i$-th observation $\mathbf{x}_i$, and that the $y_i$'s are not all the same, a $\textit{machine learning}$ approach would consist in finding the optimal parameters by minimizing the number of misclassifications, $\textit{i.e.}$ by solving 
\begin{equation*}
     \inf_{
	    	(\mathbf{w}, b) \in \mathbb{R}^{D}\times\mathbb{R}
    } \: \frac{1}{n} \sum_{i=1}^n \mathbf{1}(h_{\mathbf{w}, b}(\mathbf{x}_i) \neq y_i)\,.
\end{equation*}
However, minimizing this above objective function is tedious and one generally resorts to solving the closely related problem called the (soft) Support Vector Machine (SVM) 
$$
\mu^\ast = \inf_{ \substack{
	    	\mathbf{w} \in \mathbb{R}^{D} \\
			b \in \mathbb{R} \\
	    	\boldsymbol{\xi} \in \mathbb{R}^{n}}
    } \quad  \frac{1}{2} \|\mathbf{w}\|_2^2 + \rho \sum_{i=1}^{n} \xi_i
    \quad\text{s.t.}\quad
    \left\{
    \begin{array}{ll}
        y_i (\mathbf{w}^\top \mathbf{x_i} + b) \geq 1 - \xi_i & \forall i \in \{1, \ldots, n\} \\
        \xi_i \geq 0 & \forall i \in \{1, \ldots, n\} 
    \end{array}
\right.\,, 
$$
where $\rho > 0$ is an hyperparameter. Unless otherwise stated, we fix $\rho=1$ in the all experiments.

### 1.A. The primal problem
[Back to table of contents](#ToC)

During the previous tutorial, you have established that the primal problem could be rewritten under the following unconstrained form
$$
\mu^\ast = \inf_{ \substack{
	    	\mathbf{w} \in \mathbb{R}^{D} \\
			b \in \mathbb{R}}
    } \quad  \frac{1}{2} \|\mathbf{w}\|_2^2 + \rho \sum_{i=1}^{n} \operatorname{ReLU}(1 - y_i (\mathbf{w}^\top \mathbf{x_i} + b))
\,,
$$
where $\operatorname{ReLU}(x) = \max(0, x)$. Due to its unconstrained form, we propose to solve it using the **gradient descent** method.

**Beware:** You should notice that the regularization term is not differentiable because of the $\operatorname{ReLU}$ function. However, it is still possible to apply the **gradient descent** method by replacing the gradient by a so-called *subgradient* (see Beck, 2017) at points where the function is not differentiable. In our case, it all amounts to arbitrarily setting $\operatorname{ReLU}'(0) = 0$ (for example).

**<u>Question</u>:**
- Compute the (*sub*)gradient of the objective function with respect to $(\mathbf{w}, b) \in \mathbb{R}^{D} \times \mathbb{R}$.

$$\text{Write your answer in } \LaTeX \text{ here.}$$

### 1.B. The dual problem
[Back to table of contents](#ToC)

Using Lagrange duality, the dual problem reads
$$ 
\mu^\ast = \sup_{\boldsymbol{\lambda} \in \mathbb{R}^n 
			} \mathbf{1}^\top \boldsymbol{\lambda} -\frac{1}{2}  \| \begin{bmatrix} y_1  \mathbf{x}_1 \ldots y_n  \mathbf{x}_n  \end{bmatrix} \boldsymbol{\lambda}  \|_2^2   \quad\text{s.t.} \quad 
			\left\{
    \begin{array}{ll}
        \mathbf{y}^\top \boldsymbol{\lambda} = 0  & \\
        0 \leq \lambda_i \leq \rho & \forall i \in \{1, \ldots, n\} 
    \end{array}
\right.\,.
$$
This time, we face a constrained problem and we propose to solve it using the **projected gradient descent** (or rather **ascent** in this case) method.

Thanks to strong duality, you have established that a **minimizer** of the primal $(\mathbf{w}^\ast, b^\ast)$ can be derived via a **maximizer** of the dual $\boldsymbol{\lambda}^\ast$ as
$$
\mathbf{w}^\ast = \sum_{i=1}^n \lambda_i^\ast y_i \mathbf{x}_i  \quad \text{ and } \quad b^\ast = y_j - \mathbf{w}^{\ast \top}\mathbf{x}_j \:\: \text{ with any } j \text{ such that } 0 < \lambda_j^\ast < \rho \,.
$$

## 2. Breast Cancer Dataset
[Back to table of contents](#ToC)

In this lab, you goal is to design a binary classifier $h: \mathbb{R}^D \mapsto \{-1,+1\}$ to detect if a breast cancer is benign or malignant, given an observation $\mathbf{x} \in \mathbb{R}^D$ (also called *features*) describing the characteristics of the cell nuclei. 

The SVM approach requires a training set $\{(\mathbf{x}_i,y_i)\}_{i=1}^{n}$, where $y_i\in\{-1,+1\}$ is the label of the $i$-th observation $\mathbf{x}_i$, on which to *learn* the optimal parameters $(\mathbf{w},b)\in\mathbb{R}^{D}\times\mathbb{R}$. We propose to use the Breast Cancer Wisconsin Dataset which gathers $n=569$ samples composed of $D=31$ features labelled by experts.

In [ ]:
# Load dataset as two numpy arrays
X, y = datasets.load_breast_cancer(return_X_y=True)
y = 2*y - 1 # from {0, 1} to {-1, 1}

# Convert to DataFrame
X_df = pd.DataFrame(X, columns=datasets.load_breast_cancer().feature_names) # Convert X to a DataFrame
y_df = pd.Series(y, name='label') # Convert y to a DataFrame or Series
df = pd.concat([X_df, y_df], axis=1) # Combine X and y into one DataFrame

# Display few rows of the DataFrame
df

For the sake of simplicity, we will only consider the first two features, namely *mean radius* and *mean texture*, for now. This allows for a visual representation of the data points.

In [ ]:
# Create scatter plot
sns.scatterplot(data=df, x="mean radius", y="mean texture", hue="label", palette="viridis", alpha=0.9)

# Add labels and title
plt.xlabel("Mean radius")
plt.ylabel("Mean texture")
plt.title("Breast Cancer Data (-1: Malignent, 1: Benign)")

# Show grid
plt.grid(True, linestyle="--", alpha=0.5)

# Show plot
plt.show()

Designing a binary classifier via a linear separating hyperplane, *i.e.* of the form $h_{\mathbf{w}, b}: \mathbf{x} \in \mathbb{R}^{D} \mapsto \operatorname{sign}(\mathbf{w}^\top \mathbf{x} + b)$, seems reasonable on this dataset. Do you see why?

**<u>Questions</u>:**
1. From the above plot, give an approximate possible value for $\mathbf{w}\in \mathbb{R}^2$ and $b\in \mathbb{R}$. No calculation is expected.
2. Plot the resulting separating hyperplane on the above figure.
3. What is the misclassification rate of your handmade classifier $h_{\mathbf{w}, b}$ on the training set?
4. *(Bonus)* What is the average Euclidean distance of the data points to this linear separating hyperplane?

In [ ]:
# TODO: your code here


In the following, you will automate the computation of $b\in \mathbb{R}$ and $\mathbf{w}\in \mathbb{R}^2$ by solving the SVM optimization problem. 

## 3. Solving the primal problem with gradient descent
[Back to table of contents](#ToC)

For solving the primal problem with **gradient descent**, you will explore 3 strategies for the choice of the stepsize:
   - Constant stepsize: $\gamma_k = c_1$,
   - Variable stepsize: $\gamma_k = \frac{c_2}{1+c_3 k}$,
   - Normalized stepsize: $\gamma_k = \frac{c_4}{\|\nabla f(\mathbf{w}, b)\|_2}$,
     
where $c_1, c_2, c_3, c_4 \in \mathbb{R}$ are hyperparameters to be determined experimentally. Is there 

**<u>Questions</u>:**
1. Complete the implementation of the following functions.  
2. Plot the evolution of the objective function with the number of iterations in each case. What strategy is the best?
3. What is (approximately) the minimizer $(\mathbf{w}^\ast, b^\ast)$ of the primal problem?
4. Draw the separating hyperplane.
5. What is the misclassification rate on the training set?
6. *(Bonus)* What about $D>2$?

In [ ]:
def obj_func_primal(w, b, X, y, rho):
    """
    Computes the objective function for the primal problem.

    Parameters
    ----------
    w : np.ndarray, shape (D, )
        input vector
    b : float
        input scalar
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The hyperparameter ρ > 0.

    Returns
    -------
    float
        The value of the primal objective function at (w, b).
    """
    
def solve_primal(X, y, rho, stepsize_mode):
    """
    Solves the primal problem by gradient ascent.

    Parameters
    ----------
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The upper bound for the box constraint, with ρ > 0.
    stepsize_mode : {‘constant’, ‘variable’, 'normalized'} 
        The mode for the stepsize.

    Returns
    -------
    np.ndarray, shape (n,)
        The minimizers wopt and bopt.
    """


In [ ]:
# TODO: your code here


In [ ]:
# TODO: your code here


## 4. Solving the dual problem with projected gradient descent
[Back to table of contents](#ToC)

**<u>Questions</u>:**
1. Same questions as for the primary problem.
2. Give a lower bound as precise as possible for the primal problem.
3. *(Bonus)* A data point $\mathbf{x}_i \in \mathbb{R}^D$ is called a *support vector* if the i*th* coefficient of the dual variable $\boldsymbol{\lambda}$ is not zero. Plot the support vectors.
4. *(Bonus)* The margin is defined as the maximal Euclidean distance of a *support vector* to the separating hyperplane. Compute the margin.

We will use the function `project_onto_C` that performs the projection onto the set $\mathcal{C} = [0, \rho]^n \cap \{ \boldsymbol{\lambda} \in \mathbb{R}^n \: | \: \mathbf{y}^\top \boldsymbol{\lambda} = 0 \}$ (see exercise in the previous tutorial). Check out how to use it:

In [ ]:
help(project_onto_C)

In [ ]:
def obj_func_dual(lam, X, y, rho):
    """
    Computes the objective function for the dual problem.

    Parameters
    ----------
    lam : np.ndarray, shape (n, )
        input vector
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The hyperparameter ρ > 0.
        
    Returns
    -------
    float
        The value of the dual objective function at lam.
    """
    # TODO: your code here
    
def solve_dual(X, y, rho, stepsize_mode):
    """
    Solves the dual problem by projected gradient ascent.

    Parameters
    ----------
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The upper bound for the box constraint, with ρ > 0.
    stepsize_mode : {‘constant’, ‘variable’, 'normalized'} 
        The mode for the stepsize.

    Returns
    -------
    np.ndarray, shape (n,)
        The maximizer lamopt.
    """
    # TODO: your code here

def dual2primal(lamopt, X, y, rho):
    """
    Computes a minimizer of the primal problem, given the maximizer lamopt of the dual.

    Parameters
    ----------
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The upper bound for the box constraint, with ρ > 0.

    Returns
    -------
    Tuple(np.ndarray, float)
        The minimizers of the primal problem wopt and b.
    """
    # TODO: your code here


In [ ]:
# TODO: your code here


In [ ]:
# TODO: your code here


## 5. Bonus: Sequential minimal optimization 
[Back to table of contents](#ToC)

Sequential minimal optimization (SMO) is an alternative to projected gradient descent for solving the dual problem of SVM which does not require hard-to-tune hyperparameters such as the stepsize. Moreover, unlike gradient descent, it also has the nice property that the objective function improves at each iteration and is guaranteed to converge.

**<u>SMO algorithm</u>:**

Begin with a feasible vector $\boldsymbol{\lambda}$. SMO algorithm operates as follows:

1. Freeze all coefficients of $\boldsymbol{\lambda}$ except two taken at random: $\lambda_i$ and $\lambda_j$.
2. Update the value of $\lambda_i$ and $\lambda_j$ by solving analytically the *minimal* dual problem (there is a closed-form solution)
$$\lambda_i^\ast, \lambda_j^\ast = \arg\max_{\lambda_i, \lambda_j \in \mathbb{R} 
			} \mathbf{1}^\top \boldsymbol{\lambda} -\frac{1}{2}  \| \begin{bmatrix} y_1  \mathbf{x}_1 \ldots y_n  \mathbf{x}_n  \end{bmatrix} \boldsymbol{\lambda}  \|_2^2   \quad\text{s.t.} \quad 
			\left\{
    \begin{array}{ll}
        \mathbf{y}^\top \boldsymbol{\lambda} = 0  & \\
        0 \leq \lambda_k \leq \rho & \forall k \in \{1, \ldots, n\} 
    \end{array}
\right.\,.$$
You should notice that this problem amounts to maximizing a univariate second order polynomial with box constraints.

3. Repeat steps 1 and 2 until convergence.

**<u>Question</u>:**
Implement this algorithm.

In [ ]:
def solve_dual_smo(X, y, rho):
    """
    Solves the dual problem by SMO algorithm.

    Parameters
    ----------
    X : np.ndarray, shape (n, D)
        The features.
    y : np.ndarray, shape (n,)
        The labels.
    rho : float
        The upper bound for the box constraint, with ρ > 0.

    Returns
    -------
    np.ndarray, shape (n,)
        The maximizer lamopt.
    """
    # TODO: your code here


In [ ]:
# TODO: your code here
